In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv(r'C:\Users\sai\Desktop\learning_path_recommendor\coursera_dataset.csv')

In [3]:
df.head()

,Unnamed: 0,Title,Organization,Skills,Ratings,course_url,course_students_enrolled,course_description,Review Count,Difficulty,Type,Duration
0,0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,https://www.coursera.org/professional-certific...,"700,909",Google Cloud Fundamentals: Core Infrastructure...,20K,Beginner,Professional Certificate,3 - 6 Months
1,1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,https://www.coursera.org/professional-certific...,"229,865",Prepare for a new career in the high-growth fi...,137K,Beginner,Professional Certificate,3 - 6 Months
2,3,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,https://www.coursera.org/professional-certific...,"29,702",Prepare-se para uma nova carreira no campo de ...,100K,Beginner,Professional Certificate,3 - 6 Months
3,4,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,https://www.coursera.org/professional-certific...,"239,622",Prepare for a career in the high-growth field ...,120K,Beginner,Professional Certificate,3 - 6 Months
4,5,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,https://www.coursera.org/professional-certific...,"384,238",This course is the eighth course in the Google...,23K,Beginner,Professional Certificate,3 - 6 Months


In [4]:
df.shape

(623, 12)

In [5]:
df.columns

Index(['Unnamed: 0', 'Title', 'Organization', 'Skills', 'Ratings',
       'course_url', 'course_students_enrolled', 'course_description',
       'Review Count', 'Difficulty', 'Type', 'Duration'],
      dtype='object')

In [6]:
df.isnull().sum()

Unnamed: 0                    0
Title                         0
Organization                  0
Skills                        0
Ratings                       0
course_url                  220
course_students_enrolled    236
course_description          221
Review Count                  0
Difficulty                    0
Type                          0
Duration                      0
dtype: int64

In [7]:
df = df.drop(['Unnamed: 0','course_url','course_students_enrolled','course_description'], axis = 1)

In [8]:
df.head()

,Title,Organization,Skills,Ratings,Review Count,Difficulty,Type,Duration
0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, ...",4.8,20K,Beginner,Professional Certificate,3 - 6 Months
1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business C...",4.8,137K,Beginner,Professional Certificate,3 - 6 Months
2,Google Project Management:,Google,"Project Management, Strategy and Operations, ...",4.8,100K,Beginner,Professional Certificate,3 - 6 Months
3,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lea...",4.6,120K,Beginner,Professional Certificate,3 - 6 Months
4,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manag...",4.8,23K,Beginner,Professional Certificate,3 - 6 Months


In [9]:
df.shape

(623, 8)

In [10]:
print(df['Skills'].iloc[0])
print(df['Skills'].iloc[3])

 Network Security, Python Programming, Linux, Cloud Computing, Algorithms, Audit, Computer Programming, Computer Security Incident Management, Cryptography, Databases, Leadership and Management, Network Architecture, Risk Management, SQL
 Python Programming, Data Science, Machine Learning, Data Analysis, Algorithms, Data Management, Data Visualization, Human Learning, R Programming, Computer Programming, Data Mining, Data Structures, Database Administration, Database Application, Database Theory, Databases, Deep Learning, Exploratory Data Analysis, Machine Learning Algorithms, Plot (Graphics), SQL, Data Model, Statistical Machine Learning, General Statistics, Probability & Statistics, Regression, Reinforcement Learning, Statistical Programming, Big Data, Cloud Computing, IBM Cloud, Writing


In [11]:
vectorizer = TfidfVectorizer()
skill_vectors = vectorizer.fit_transform(df['Skills'])


In [12]:
user_skills = ['python', 'sql', 'machine learning']
user_skills_string = ', '.join(user_skills)
user_vector = vectorizer.transform([user_skills_string])
print(user_vector.shape)

(1, 314)


In [13]:
similarities = cosine_similarity(user_vector,skill_vectors).flatten()
print(similarities.shape)
print(similarities[:5])

(623,)
[0.18028466 0.06999871 0.         0.41345349 0.        ]


In [14]:
result = df.copy()
result['similarity_score'] = similarities
top_courses = result.sort_values('similarity_score',ascending = False).head(10)
print(top_courses[['Title','Difficulty','similarity_score']])

                                                 Title    Difficulty  \
410  Ciclo completo del desarrollo de un proyecto d...      Beginner   
369             Applied Data Science for Data Analysts  Intermediate   
519  Applying Machine Learning to your Data with Go...      Beginner   
46   Supervised Machine Learning: Regression and Cl...      Beginner   
440  Prepare for DP-100: Data Science on Microsoft ...  Intermediate   
288                  Deep Neural Networks with PyTorch  Intermediate   
271        Convolutional Neural Networks in TensorFlow  Intermediate   
87                        Advanced Learning Algorithms      Beginner   
258                 Applied Machine Learning in Python  Intermediate   
7                                     Machine Learning      Beginner   

     similarity_score  
410          0.683031  
369          0.673087  
519          0.661380  
46           0.654281  
440          0.637080  
288          0.624731  
271          0.621104  
87           0.

In [15]:
user_level = 'Beginner'
filtered = result[result['Difficulty'] == user_level]
top_filtered = filtered.sort_values('similarity_score', ascending = False).head(5)

In [16]:
print(top_filtered)

                                                 Title  \
410  Ciclo completo del desarrollo de un proyecto d...   
519  Applying Machine Learning to your Data with Go...   
46   Supervised Machine Learning: Regression and Cl...   
87                        Advanced Learning Algorithms   
7                                     Machine Learning   

                 Organization  \
410  Coursera Project Network   
519              Google Cloud   
46            DeepLearning.AI   
87            DeepLearning.AI   
7          Multiple educators   

                                                Skills  Ratings Review Count  \
410                                   Machine Learning      4.8           32   
519   Applied Machine Learning, Big Data, Cloud Com...      4.7         1.1K   
46    Machine Learning, Machine Learning Algorithms...      4.9          16K   
87    Applied Machine Learning, Machine Learning, M...      4.9         4.3K   
7     Machine Learning, Machine Learning Algorithms.

In [17]:
def recommended_courses(user_skills, user_level, top_n = 5):
    user_skills_string = ', '.join(user_skills)
    user_vector = vectorizer.transform([user_skills_string])
    
    similarities = cosine_similarity(user_vector,skill_vectors).flatten()
    
    result = df.copy()
    result['similarity_score'] = similarities
    
    filtered = result[result['Difficulty'] == user_level]
    top_filtered = filtered.sort_values('similarity_score', ascending = False).head(5)
    
    return top_filtered[['Title', 'Organization', 'Difficulty', 'Ratings', 'similarity_score']]

In [18]:
recommended_courses(['Python', 'Data Analysis'], 'Beginner')

,Title,Organization,Difficulty,Ratings,similarity_score
376,Exploratory Data Analysis With Python and Pandas,Coursera Project Network,Beginner,4.5,0.775307
363,Python for Data Analysis: Pandas & NumPy,Coursera Project Network,Beginner,4.4,0.747311
134,ChatGPT Advanced Data Analysis,Vanderbilt University,Beginner,4.9,0.708077
397,Data Analysis Using Python,University of Pennsylvania,Beginner,4.5,0.613275
108,"Python for Data Science, AI & Development",IBM,Beginner,4.6,0.604546
